# 10주 · 비에너지·비력·도수

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 비에너지·한계류 / 비력·수문 힘·도수

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week10/CE272_W10.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week10) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 비에너지 참조식·수문 힘·도수

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 비에너지 곡선과 원본 Excel 참조식

**슬라이드 연결**: CE272_W10a_Steady_Nonuniform_Flow_1.pptx p3, 5, 6, 7, 8, 9, 10, 11

**자료 구분**: 원본 비에너지 Excel·참조식 대조

**먼저 예측**: 유량을 바꾼 비에너지 곡선도 같은 수심 좌표를 참조해야 하는가?

**문제**: 단위 폭 유량 q=1과 1.5 m²/s에 대해 E=y+q²/(2gy²)를 계산하시오. 원본 Excel C열의 B열 참조와 올바른 A열 수심 참조가 만드는 차이를 설명하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
ys=np.linspace(.09,5.02,494)
E1=ys+1**2/(2*hyd.G*ys**2)
E15=ys+1.5**2/(2*hyd.G*ys**2)
legacy=E1+1.5**2/(2*hyd.G*E1**2)  # 원본 C열의 참조를 재현한 비교값
for q in [1.,1.5]:
    yc=(q*q/hyd.G)**(1/3)
    print('q, yc, Emin:',q,yc,1.5*yc)
plt.plot(E1,ys,label='q=1');plt.plot(E15,ys,label='q=1.5, correct')
plt.plot(legacy,ys,'--',label='q=1.5, original cell reference')
plt.xlabel('E (m)');plt.ylabel('y (m)');plt.xlim(0,6);plt.legend();plt.grid();plt.show()

**검토 문제**: 최솟값에서 Fr=1인지 확인하시오. 그래프 모양이 그럴듯해도 셀 참조를 검토해야 하는 이유를 단위·변수 정의로 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 수문에 작용하는 힘

**슬라이드 연결**: CE272_W10b_Steady_Nonuniform_Flow_2.pptx p12, 13, 14, 15, 16, 17

**자료 구분**: 원본 예제 7.5

**먼저 예측**: 수압 합력 차가 전부 수문에 전달되는가?

**문제**: 폭 5 m의 직사각형 수로에서 Q=14 m³/s, 상류 수심 2 m, 하류 수심 0.5 m이다. 물이 수문에 가하는 수평력의 크기와 방향을 구하시오. 물의 밀도는 1,000 kg/m³이다.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
b,Q,y1,y2,rho=5.,14.,2.,.5,1000.
M1,M2=hyd.specific_force(Q,y1,b),hyd.specific_force(Q,y2,b)
F_water_on_gate=rho*hyd.G*(M1-M2)
pressure_difference=rho*hyd.G*b*(y1*y1-y2*y2)/2
momentum_change=rho*Q*(Q/(b*y2)-Q/(b*y1))
pd.Series({'M1_m3':M1,'M2_m3':M2,'water_on_gate_kN':F_water_on_gate/1000,
           'pressure_difference_kN':pressure_difference/1000,
           'momentum_change_kN':momentum_change/1000,
           'force_balance_residual_N':pressure_difference-momentum_change-F_water_on_gate})

**검토 문제**: 물의 검사체적에 작용하는 힘과 물이 수문에 가하는 힘의 방향을 구분하시오. 원본 반올림 정답 약 33.2 kN과 비교하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S03. 공액수심과 에너지 손실

**슬라이드 연결**: CE272_W10b_Steady_Nonuniform_Flow_2.pptx p8, 18, 19, 20, 21, 22

**자료 구분**: 원본 도수 원리·앞 예제 조건 확장

**먼저 예측**: 수문 예제의 두 수심을 외력이 없는 도수의 공액수심으로 볼 수 있는가?

**문제**: 폭 5 m, Q=14 m³/s, 도수 전 수심 0.5 m인 직사각형 수로의 공액수심을 계산하시오. 비력과 비에너지를 각각 비교하고, 앞 수문 예제의 수심 2 m와 구분하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
b,Q,y_before=5.,14.,.5
y_after=hyd.conjugate_depth(Q,y_before,b)
E_before=hyd.specific_energy(Q,y_before,b);E_after=hyd.specific_energy(Q,y_after,b)
loss_exact=(y_after-y_before)**3/(4*y_before*y_after)
pd.Series({'conjugate_depth_m':y_after,'Fr_before':hyd.froude(Q,y_before,b),
           'Fr_after':hyd.froude(Q,y_after,b),'energy_loss_m':E_before-E_after,
           'jump_loss_formula_m':loss_exact,
           'momentum_function_residual_m3':hyd.specific_force(Q,y_before,b)-hyd.specific_force(Q,y_after,b)})

**검토 문제**: 도수에서 비력은 같고 비에너지는 감소하는 조건을 설명하시오. 외력과 손실이 있는 문제에 잘못된 보존식을 적용한 사례를 작성하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

에너지 방정식과 운동량 방정식의 선택 근거, 외력·손실과 수심의 관계를 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')